# Linear Regression — Real Estate.csv
Predict **house price per unit area** from transaction date, house age, distance to MRT, number of convenience stores, latitude, longitude.

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("Real_estate.csv")
df.head()

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1


In [2]:
df.shape

(414, 8)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 8 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   No                                      414 non-null    int64  
 1   X1 transaction date                     414 non-null    float64
 2   X2 house age                            414 non-null    float64
 3   X3 distance to the nearest MRT station  414 non-null    float64
 4   X4 number of convenience stores         414 non-null    int64  
 5   X5 latitude                             414 non-null    float64
 6   X6 longitude                            414 non-null    float64
 7   Y house price of unit area              414 non-null    float64
dtypes: float64(6), int64(2)
memory usage: 26.0 KB


## 2. Preprocessing

In [4]:
# Drop the "No" serial-number column (not a predictive feature)
if "No" in df.columns:
    df = df.drop(columns=["No"])

print("Missing values:\n", df.isna().sum())
df = df.dropna()

Missing values:
 X1 transaction date                       0
X2 house age                              0
X3 distance to the nearest MRT station    0
X4 number of convenience stores           0
X5 latitude                               0
X6 longitude                              0
Y house price of unit area                0
dtype: int64


In [5]:
target_col = "Y house price of unit area"
X = df.drop(columns=[target_col])
y = df[target_col]
feature_columns = X.columns.tolist()
feature_columns

['X1 transaction date',
 'X2 house age',
 'X3 distance to the nearest MRT station',
 'X4 number of convenience stores',
 'X5 latitude',
 'X6 longitude']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

Train shape: (331, 6)  Test shape: (83, 6)


## 3. Model Training

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Coefficients:", dict(zip(X.columns, model.coef_)))
print("Intercept:", model.intercept_)

Coefficients: {'X1 transaction date': np.float64(5.440741856695847), 'X2 house age': np.float64(-0.27079148996249625), 'X3 distance to the nearest MRT station': np.float64(-0.004758638917293966), 'X4 number of convenience stores': np.float64(1.091425267691966), 'X5 latitude': np.float64(229.04305381550014), 'X6 longitude': np.float64(-29.492590776198153)}
Intercept: -13044.231917160534


## 4. Model Evaluation — MAE & R² Score

In [8]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Absolute Error (MAE): 5.3054
R2 Score: 0.6811


In [9]:
pd.DataFrame({"Actual": y_test, "Predicted": y_pred}).head(10)

,Actual,Predicted
358,45.1,47.886254
350,42.3,41.164046
373,52.2,44.273014
399,37.3,40.197615
369,22.8,27.513265
72,36.3,45.109531
262,53.0,44.632933
140,51.4,46.363462
93,16.1,23.620631
70,59.0,54.334449


## 5. Predictive Function

In [10]:
def predict_price(transaction_date, house_age, distance_to_mrt,
                   num_convenience_stores, latitude, longitude):
    """
    Predict house price per unit area for a new property.
    """
    input_df = pd.DataFrame([{
        "X1 transaction date": transaction_date,
        "X2 house age": house_age,
        "X3 distance to the nearest MRT station": distance_to_mrt,
        "X4 number of convenience stores": num_convenience_stores,
        "X5 latitude": latitude,
        "X6 longitude": longitude
    }])
    input_df = input_df.reindex(columns=feature_columns, fill_value=0)
    prediction = model.predict(input_df)[0]
    return prediction

## 6. Prediction (example)

In [11]:
predicted_price = predict_price(
    transaction_date=2013.25, house_age=10.0, distance_to_mrt=500.0,
    num_convenience_stores=5, latitude=24.98, longitude=121.54
)
print(f"Predicted price per unit area: {predicted_price:.2f}")

Predicted price per unit area: 46.68
